# Please refer to the BANYAN Σ package:
https://github.com/jgagneastro/banyan_sigma

Installing it in a separate Python environment (e.g., using conda) is recommended to avoid package conflicts.

The following packages are also needed: mocapy, pandas, multiprocess, tqdm

## BANYAN $\Sigma$ for single stars

In [51]:
# Pleiades candidate: HIP 17034 (HD 22614)

from banyan_sigma import *

# Data from BANYAN Sigma on https://mocadb.ca/

ra=     54.77812109736227       # deg
dec=    24.702658564560203      # deg
pmra=   21.688600540161133	    # mas/yr
epmra=  0.03835409879684448     # mas/yr
pmdec=  -47.25979995727539      # mas/yr
epmdec= 0.02764499932527542     # mas/yr
plx=    7.702829837799072       # mas
eplx=   0.033775098621845245    # mas
rv=     4.367651                # km/s
erv=    1.83236                 # km/s


#Determine membership probability
output = membership_probability(
    ra=ra,dec=dec,pmra=pmra,pmdec=pmdec,epmra=epmra,epmdec=epmdec,plx=plx,eplx=eplx,rv=rv,erv=erv, use_plx=True, use_rv=True
)

In [52]:
# The membership probability of the Pleiades
output['ALL']['PLE'] # Gaia

0    0.999663
Name: PLE, dtype: float64

In [44]:
# The complete membership probability
output['ALL'].max()

118TAU    3.970795e-184
ABDMG      7.241132e-10
BPMG       3.323816e-48
CAR       1.776425e-168
CARN       5.304056e-98
CBER       0.000000e+00
COL       1.974128e-199
CRA        0.000000e+00
EPSC       0.000000e+00
ETAC       0.000000e+00
HYA        0.000000e+00
IC2391     0.000000e+00
IC2602     0.000000e+00
LCC        1.247950e-60
OCT        0.000000e+00
PL8        0.000000e+00
PLE        9.996630e-01
ROPH       0.000000e+00
TAU        9.872388e-12
THA       3.762144e-102
THOR      1.716681e-108
TWA       4.006595e-224
UCL        4.210102e-55
UCRA       0.000000e+00
UMA        0.000000e+00
USCO      1.291469e-118
XFOR       0.000000e+00
VCA        0.000000e+00
ARG        2.085892e-39
PRAE       0.000000e+00
MUTAU      3.122633e-20
PERI      4.332559e-129
FIELD      3.369744e-04
dtype: float64

In [ ]:
# Investigate the outputs
output.iloc[0]

## BANYAN $\Sigma$ for cluster members

In [5]:
from mocapy import MocaEngine


def get_star_kinematics(df, column=""):
    """
    Query astrometry and radial velocities from MOCAdb
    using stellar designations from a column in the input DataFrame.

    Parameters
    ----------
    df : pandas.DataFrame
        Input DataFrame containing a column with stellar designations.

    column : str
        Name of the column containing stellar designations.

    Returns
    -------
    pandas.DataFrame
        Original DataFrame with MOCAdb results appended using
        the following column names:

        ra_banyan, dec_banyan,
        pmra_banyan, epmra_banyan,
        pmdec_banyan, epmdec_banyan,
        plx_banyan, eplx_banyan,
        rv_banyan, erv_banyan
    """

    if column not in df.columns:
        raise ValueError(
            f"Column '{column}' not found in DataFrame."
        )

    moca = MocaEngine()

    input_df = pd.DataFrame({
        "designation": df[column].values
    })

    query = """
    SELECT
        tt.designation AS designation,

        mo.ra AS ra,
        mo.dec AS `dec`,

        pm.pmra_masyr AS pmra,
        pm.pmra_masyr_unc AS epmra,
        pm.pmdec_masyr AS pmdec,
        pm.pmdec_masyr_unc AS epmdec,

        plx.parallax_mas AS plx,
        plx.parallax_mas_unc AS eplx,

        rv.radial_velocity_kms AS rv,
        rv.radial_velocity_kms_unc AS erv

    FROM tmp_table AS tt

    LEFT JOIN mechanics_all_designations AS mad
        ON mad.designation = tt.designation

    LEFT JOIN moca_objects AS mo
        ON mo.moca_oid = mad.moca_oid

    LEFT JOIN data_proper_motions AS pm
        ON pm.moca_oid = mad.moca_oid
        AND pm.adopted = 1

    LEFT JOIN data_parallaxes AS plx
        ON plx.moca_oid = mad.moca_oid
        AND plx.adopted = 1

    LEFT JOIN calc_radial_velocities_combined AS rv
        ON rv.moca_oid = mad.moca_oid
    """

    moca_df = moca.query(
        query,
        tmp_table=input_df
    )

    # Only keep the columns obtained from MOCAdb.
    moca_columns = [
        "ra",
        "dec",
        "pmra",
        "epmra",
        "pmdec",
        "epmdec",
        "plx",
        "eplx",
        "rv",
        "erv",
    ]

    moca_df = moca_df[moca_columns].copy()

    # Add "_banyan" suffix to distinguish these values
    # from the original DataFrame columns.
    moca_df = moca_df.rename(
        columns={
            col: f"{col}_banyan"
            for col in moca_columns
        }
    )

    # Reset indices before concatenating so that the rows correspond
    # to the original input DataFrame.
    moca_df = moca_df.reset_index(drop=True)
    df_output = df.reset_index(drop=True).copy()

    # Append MOCAdb results to the original DataFrame.
    df_output = pd.concat(
        [df_output, moca_df],
        axis=1
    )

    return df_output

In [6]:
import pandas as pd
from multiprocess import Pool
from tqdm import tqdm
from banyan_sigma import membership_probability


def _banyan_one_star(row):
    try:
        use_plx = pd.notna(row["plx_banyan"]) and pd.notna(row["eplx_banyan"])
        use_rv = pd.notna(row["rv_banyan"]) and pd.notna(row["erv_banyan"])

        output = membership_probability(
            ra=row["ra_banyan"],
            dec=row["dec_banyan"],
            pmra=row["pmra_banyan"],
            pmdec=row["pmdec_banyan"],
            epmra=row["epmra_banyan"],
            epmdec=row["epmdec_banyan"],
            plx=row["plx_banyan"] if use_plx else None,
            eplx=row["eplx_banyan"] if use_plx else None,
            rv=row["rv_banyan"] if use_rv else None,
            erv=row["erv_banyan"] if use_rv else None,
            use_plx=use_plx,
            use_rv=use_rv,
        )

        return {
            "success": True,
            "result": output["ALL"].iloc[0].to_numpy(),
            "error": None,
        }

    except Exception as e:
        return {
            "success": False,
            "result": None,
            "error": str(e),
        }


def add_banyan_probabilities(df, n_processes=8):

    required_columns = [
        "ra_banyan",
        "dec_banyan",
        "pmra_banyan",
        "epmra_banyan",
        "pmdec_banyan",
        "epmdec_banyan",
        "plx_banyan",
        "eplx_banyan",
        "rv_banyan",
        "erv_banyan",
    ]

    missing = [
        col for col in required_columns
        if col not in df.columns
    ]

    if missing:
        raise ValueError(
            f"Missing required columns: {missing}"
        )

    df_input = df.copy()

    # Only pass the columns required by BANYAN.
    # This avoids problems with duplicate column names
    # elsewhere in the input DataFrame.
    rows = df_input[required_columns].to_dict("records")

    # -------------------------
    # Run BANYAN
    # -------------------------
    results = []

    with Pool(processes=n_processes) as pool:
        for result in tqdm(
            pool.imap(_banyan_one_star, rows),
            total=len(rows),
            desc="BANYAN",
        ):
            results.append(result)

    # -------------------------
    # BANYAN columns
    # -------------------------
    banyan_columns = [
        "118TAU", "ABDMG", "BPMG", "CAR", "CARN", "CBER", "COL",
        "CRA", "EPSC", "ETAC", "HYA", "IC2391", "IC2602", "LCC",
        "OCT", "PL8", "PLE", "ROPH", "TAU", "THA", "THOR", "TWA",
        "UCL", "UCRA", "UMA", "USCO", "XFOR", "VCA", "ARG", "PRAE",
        "MUTAU", "PERI", "FIELD"
    ]

    # -------------------------
    # Construct BANYAN DataFrame
    # -------------------------
    banyan_data = []

    for r in results:
        if r["success"]:
            banyan_data.append(r["result"])
        else:
            banyan_data.append(
                [float("nan")] * len(banyan_columns)
            )

    banyan_df = pd.DataFrame(
        banyan_data,
        columns=banyan_columns,
        index=df_input.index,
    )

    # -------------------------
    # Failed stars
    # -------------------------
    failed_positions = [
        i for i, r in enumerate(results)
        if not r["success"]
    ]

    if failed_positions:

        failed_indices = df_input.index[failed_positions]

        print("\nFailed BANYAN indices:")
        print(list(failed_indices))

        if "designation" in df_input.columns:
            for pos in failed_positions:
                print(
                    f"  index={df_input.index[pos]}, "
                    f"designation={df_input.iloc[pos]['designation']}, "
                    f"error={results[pos]['error']}"
                )

        df_failed = df_input.iloc[failed_positions].copy()

        df_failed["error"] = [
            results[i]["error"]
            for i in failed_positions
        ]

    else:

        print("\nNo BANYAN failures.")

        df_failed = pd.DataFrame(
            columns=list(df_input.columns) + ["error"]
        )

    # -------------------------
    # Final result
    # -------------------------
    df_output = pd.concat(
        [df_input, banyan_df],
        axis=1,
    )

    return df_output, df_failed

In [9]:
id_columns = [
    "HIP_ID",
    "HD_ID",
    "Gaia_DR3_ID",
    "TYC_ID",
    "TwoMass_ID",
    "Other_Name",
]

In [17]:
df_hyades = pd.read_csv('./vizier_J_A+A_623_A35/Hyades_rough_selection_with_query_data_gtomo_vsini.csv', 
                        dtype={col: "string" for col in id_columns})
df_hyades_banyan_data = get_star_kinematics(df_hyades, column='target_name')
df_hyades_banyan_results, df_hyades_banyan_failed = add_banyan_probabilities(df_hyades_banyan_data,n_processes=8)
df_hyades_banyan_results.to_csv('./vizier_J_A+A_623_A35/Hyades_rough_selection_with_query_data_gtomo_vsini_banyan.csv', index=True)
print(df_hyades_banyan_failed)

# one failed star will be dropped:
# Gaia DR2 478368788803049600: Spectroscopic Binary


    SELECT
        tt.designation AS designation,

        mo.ra AS ra,
        mo.dec AS `dec`,

        pm.pmra_masyr AS pmra,
        pm.pmra_masyr_unc AS epmra,
        pm.pmdec_masyr AS pmdec,
        pm.pmdec_masyr_unc AS epmdec,

        plx.parallax_mas AS plx,
        plx.parallax_mas_unc AS eplx,

        rv.radial_velocity_kms AS rv,
        rv.radial_velocity_kms_unc AS erv

    FROM tmp_table_46e92ddfd6cf41fe973cf2b9f4eaca29 AS tt

    LEFT JOIN mechanics_all_designations AS mad
        ON mad.designation = tt.designation

    LEFT JOIN moca_objects AS mo
        ON mo.moca_oid = mad.moca_oid

    LEFT JOIN data_proper_motions AS pm
        ON pm.moca_oid = mad.moca_oid
        AND pm.adopted = 1

    LEFT JOIN data_parallaxes AS plx
        ON plx.moca_oid = mad.moca_oid
        AND plx.adopted = 1

    LEFT JOIN calc_radial_velocities_combined AS rv
        ON rv.moca_oid = mad.moca_oid
    


BANYAN: 100%|█████████████████████████████████████████████████████████████████████████████████████| 57/57 [00:27<00:00,  2.07it/s]


Failed BANYAN indices:
[17]
                    target_name  recno                  ID     RAICRS  \
17  Gaia DR2 478368788803049600    472  478368788803049600  77.678219   

       DEICRS    pmRA*     pmDE      Plx    Gmag      bx  ...  dec_banyan  \
17  63.596864  34.4051 -82.4418  11.0991  6.6696 -73.988  ...         NaN   

    pmra_banyan  epmra_banyan  pmdec_banyan  epmdec_banyan  plx_banyan  \
17          NaN           NaN           NaN            NaN         NaN   

    eplx_banyan  rv_banyan erv_banyan  \
17          NaN        NaN        NaN   

                                                error  
17  The observables ra,dec,pmra,pmdec,epmra and ep...  

[1 rows x 88 columns]


In [15]:
df_praesepe = pd.read_csv('./vizier_J_A+A_628_A66/Praesepe_rough_selection_with_query_data_gtomo_vsini.csv', 
                          dtype={col: "string" for col in id_columns})
df_praesepe_banyan_data = get_star_kinematics(df_praesepe, column='target_name')
df_praesepe_banyan_results, df_praesepe_banyan_failed = add_banyan_probabilities(df_praesepe_banyan_data,n_processes=8)
df_praesepe_banyan_results.to_csv('./vizier_J_A+A_628_A66/Praesepe_rough_selection_with_query_data_gtomo_vsini_banyan.csv', index=True)
print(df_praesepe_banyan_failed)

# two failed stars will be dropped:
# Gaia DR2 682570706786179072 (HD 69477): high HGCA acceleration
# Gaia DR2 608783026905699328: MBP-MRP=0.59>0.47, which will be exluded later


    SELECT
        tt.designation AS designation,

        mo.ra AS ra,
        mo.dec AS `dec`,

        pm.pmra_masyr AS pmra,
        pm.pmra_masyr_unc AS epmra,
        pm.pmdec_masyr AS pmdec,
        pm.pmdec_masyr_unc AS epmdec,

        plx.parallax_mas AS plx,
        plx.parallax_mas_unc AS eplx,

        rv.radial_velocity_kms AS rv,
        rv.radial_velocity_kms_unc AS erv

    FROM tmp_table_f8504653e74a48ecaa4f521036639beb AS tt

    LEFT JOIN mechanics_all_designations AS mad
        ON mad.designation = tt.designation

    LEFT JOIN moca_objects AS mo
        ON mo.moca_oid = mad.moca_oid

    LEFT JOIN data_proper_motions AS pm
        ON pm.moca_oid = mad.moca_oid
        AND pm.adopted = 1

    LEFT JOIN data_parallaxes AS plx
        ON plx.moca_oid = mad.moca_oid
        AND plx.adopted = 1

    LEFT JOIN calc_radial_velocities_combined AS rv
        ON rv.moca_oid = mad.moca_oid
    


BANYAN: 100%|█████████████████████████████████████████████████████████████████████████████████████| 71/71 [00:32<00:00,  2.17it/s]


Failed BANYAN indices:
[2, 69]
                    target_name  recno      RA_ICRS      DE_ICRS  \
2   Gaia DR2 682570706786179072    223  08 18 06.57  +26 25 28.4   
69  Gaia DR2 608783026905699328   1980  08 54 51.31  +14 29 44.1   

              SourceCP   RVCP  e_RVCP    PlxCP   e_PlxCP  GmagCP  ...  \
2   682570706786179072    NaN     NaN  5.16086  0.049384   7.165  ...   
69  608783026905699328  31.12    0.57  4.84527  0.042707   9.262  ...   

    dec_banyan  pmra_banyan  epmra_banyan  pmdec_banyan  epmdec_banyan  \
2          NaN          NaN           NaN           NaN            NaN   
69         NaN          NaN           NaN           NaN            NaN   

    plx_banyan  eplx_banyan  rv_banyan  erv_banyan  \
2          NaN          NaN        NaN         NaN   
69         NaN          NaN        NaN         NaN   

                                                error  
2   The observables ra,dec,pmra,pmdec,epmra and ep...  
69  The observables ra,dec,pmra,pmdec,epmra a

In [16]:
df_pleiades = pd.read_csv('./vizier_J_A+A_628_A66/Pleiades_rough_selection_with_query_data_gtomo_vsini.csv', 
                          dtype={col: "string" for col in id_columns})
df_pleiades_banyan_data = get_star_kinematics(df_pleiades, column='target_name')
df_pleiades_banyan_results, df_pleiades_banyan_failed = add_banyan_probabilities(df_pleiades_banyan_data,n_processes=8)
df_pleiades_banyan_results.to_csv('./vizier_J_A+A_628_A66/Pleiades_rough_selection_with_query_data_gtomo_vsini_banyan.csv', index=True)
print(len(df_pleiades_banyan_failed))

# no failed target


    SELECT
        tt.designation AS designation,

        mo.ra AS ra,
        mo.dec AS `dec`,

        pm.pmra_masyr AS pmra,
        pm.pmra_masyr_unc AS epmra,
        pm.pmdec_masyr AS pmdec,
        pm.pmdec_masyr_unc AS epmdec,

        plx.parallax_mas AS plx,
        plx.parallax_mas_unc AS eplx,

        rv.radial_velocity_kms AS rv,
        rv.radial_velocity_kms_unc AS erv

    FROM tmp_table_442883e8f3e544dfb620f23d2c8de21c AS tt

    LEFT JOIN mechanics_all_designations AS mad
        ON mad.designation = tt.designation

    LEFT JOIN moca_objects AS mo
        ON mo.moca_oid = mad.moca_oid

    LEFT JOIN data_proper_motions AS pm
        ON pm.moca_oid = mad.moca_oid
        AND pm.adopted = 1

    LEFT JOIN data_parallaxes AS plx
        ON plx.moca_oid = mad.moca_oid
        AND plx.adopted = 1

    LEFT JOIN calc_radial_velocities_combined AS rv
        ON rv.moca_oid = mad.moca_oid
    


BANYAN: 100%|█████████████████████████████████████████████████████████████████████████████████████| 76/76 [00:44<00:00,  1.70it/s]


No BANYAN failures.
0
